In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import gradio as gr

from transformers import (
    DistilBertTokenizer,
    DistilBertModel,
    AutoTokenizer,
    AutoModelForSequenceClassification
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [3]:
BASE_PATH = "/content/drive/MyDrive/phishing_models"


In [4]:
class BERTEmailClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        cls = self.dropout(cls)
        return self.fc(cls)


In [5]:
tokenizer = DistilBertTokenizer.from_pretrained(
    f"{BASE_PATH}/email",
    local_files_only=True
)

email_model = BERTEmailClassifier().to(device)

email_model.load_state_dict(
    torch.load(
        f"{BASE_PATH}/email/email_model.pt",
        map_location=device
    )
)

email_model.eval()
print("Email model loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Email model loaded


In [6]:
url_tokenizer = AutoTokenizer.from_pretrained(
    f"{BASE_PATH}/url",
    local_files_only=True
)

url_model = AutoModelForSequenceClassification.from_pretrained(
    f"{BASE_PATH}/url",
    local_files_only=True
)

url_model.to(device)
url_model.eval()
print("URL model loaded")


URL model loaded


In [7]:
LEAKAGE_PATTERNS = [
    r"\bspam\b",
    r"\bphish\b",
    r"\bphishing\b",
    r"\bham\b"
]

def remove_leakage(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    for p in LEAKAGE_PATTERNS:
        text = re.sub(p, "", text)
    return text


In [8]:
def predict(subject, body, url):
    outputs = []

    # -------- Email prediction --------
    if subject.strip() or body.strip():
        text = remove_leakage(subject) + " [SEP] " + remove_leakage(body)

        enc = tokenizer(
            [text],
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            logits = email_model(enc["input_ids"], enc["attention_mask"])
            probs = F.softmax(logits, dim=1)[0]

        label = "Phishing" if probs[1] > 0.5 else "Legitimate"
        confidence = float(probs[1] if label == "Phishing" else probs[0])

        outputs.append(f"Email Prediction: {label} ({confidence*100:.2f}%)")
    else:
        outputs.append("Email Prediction: No email provided")

    # -------- URL prediction --------
    if url.strip():
        inputs = url_tokenizer(
            [url],
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = url_model(**inputs).logits
            probs = F.softmax(logits, dim=1)[0]

        label = "Phishing" if probs[1] > 0.5 else "Legitimate"
        confidence = float(probs[1] if label == "Phishing" else probs[0])

        outputs.append(f"URL Prediction: {label} ({confidence*100:.2f}%)")

    return "\n".join(outputs)


In [9]:
interface = gr.Interface(
    fn=predict,
    inputs=[
        gr.Textbox(label="Email Subject"),
        gr.Textbox(lines=8, label="Email Body"),
        gr.Textbox(label="URL (optional)")
    ],
    outputs=gr.Textbox(label="Predictions"),
    title="Phishing Detection System",
    description="Enter an email subject & body and/or a URL to check if it is phishing."
)

interface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5fb68e558e24dd9110.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
